In [1]:
model_path = "/mnt/petrelfs/huzican/R1/rlvr_div/checkpoints/div/baseline_passk_training/actor/global_step_350"
# model_path = "/mnt/petrelfs/share_data/zhangshilin/entropy_clip_cov"
# model_path = "/mnt/petrelfs/share_data/zhangshilin/rs_equ_100step"
# model_path = "/mnt/petrelfs/share_data/zhangshilin/baseline_clp_02_028_300"

In [2]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "1" 

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from verl.utils.dataset.rl_dataset import RLHFDataset, collate_fn
from torch.utils.data import DataLoader
from vllm import LLM, SamplingParams

In [3]:

tokenizer = AutoTokenizer.from_pretrained(model_path)
torch.cuda.empty_cache()
base_model = LLM(
    model=model_path,
    tensor_parallel_size=1,
    gpu_memory_utilization=0.85,
    dtype="auto"
)

INFO 09-22 11:44:22 config.py:1450] Downcasting torch.float32 to torch.float16.
INFO 09-22 11:44:22 llm_engine.py:174] Initializing an LLM engine (v0.5.4) with config: model='/mnt/petrelfs/huzican/R1/rlvr_div/checkpoints/div/baseline_passk_training/actor/global_step_350', speculative_config=None, tokenizer='/mnt/petrelfs/huzican/R1/rlvr_div/checkpoints/div/baseline_passk_training/actor/global_step_350', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, rope_scaling=None, rope_theta=None, tokenizer_revision=None, trust_remote_code=False, dtype=torch.float16, max_seq_len=16384, download_dir=None, load_format=LoadFormat.AUTO, tensor_parallel_size=1, pipeline_parallel_size=1, disable_custom_all_reduce=False, quantization=None, enforce_eager=False, kv_cache_dtype=auto, quantization_param_path=None, device_config=cuda, decoding_config=DecodingConfig(guided_decoding_backend='outlines'), observability_config=ObservabilityConfig(otlp_traces_endpoint=None), seed=0, served_model_name

Loading safetensors checkpoint shards:   0% Completed | 0/7 [00:00<?, ?it/s]


INFO 09-22 11:44:36 model_runner.py:732] Loading model weights took 14.2448 GB
INFO 09-22 11:44:38 gpu_executor.py:102] # GPU blocks: 59353, # CPU blocks: 4681
INFO 09-22 11:44:41 model_runner.py:1024] Capturing the model for CUDA graphs. This may lead to unexpected consequences if the model is not static. To run the model in eager mode, set 'enforce_eager=True' or use '--enforce-eager' in the CLI.
INFO 09-22 11:44:41 model_runner.py:1028] CUDA graphs can take additional 1~3 GiB memory per GPU. If you are running out of memory, consider decreasing `gpu_memory_utilization` or enforcing eager mode. You can also reduce the `max_num_seqs` as needed to decrease memory usage.
INFO 09-22 11:44:55 model_runner.py:1225] Graph capturing finished in 14 secs.


In [4]:
val_data_path = "dataset/eval.passn.parquet"
val_dataset = RLHFDataset(parquet_files=val_data_path,
                            tokenizer=tokenizer,
                            prompt_key='prompt',
                            max_prompt_length=1024,
                            filter_prompts=True,
                            return_raw_chat=False,
                            truncation='error')

val_dataloader = DataLoader(dataset=val_dataset,
                            batch_size=128,
                            shuffle=False,
                            drop_last=False,
                            collate_fn=collate_fn)
n_val_samples = 8

original dataset len: 1590
filter dataset len: 1588


In [10]:
data_idx = 1000
test_data = val_dataset[data_idx]
# print(test_data['reward_model'])
input_text = tokenizer.decode(test_data['input_ids'], skip_special_tokens=True)
sampling_params = SamplingParams(
    temperature=0.6,
    top_p=1.0,
    max_tokens=8192,
)



In [11]:
prompts = [input_text] * n_val_samples
outputs = base_model.generate(prompts, sampling_params)

Processed prompts: 100%|██████████| 8/8 [00:16<00:00,  2.06s/it, est. speed input: 135.51 toks/s, output: 535.50 toks/s]


In [12]:
group_rollout = []
for output in outputs:
    # 提取生成的文本
    full_text = output.outputs[0].text
    # 只保留输入之后新生成的部分
    generated_text = full_text[len(input_text):]
    group_rollout.append(generated_text)

In [13]:
print(test_data['reward_model'])
for i in range(len(group_rollout)):
    print(f"********{i}**********")
    print(group_rollout[i])  # 从列表中提取文本
    print("__________end_____________\n")

{'ground_truth': '12', 'style': 'rule'}
********0**********
ll $x_i = 0$, which is a trivial solution. If $a \neq 0$, then $a^{k-2} = 1$. This means that $a$ must be a $(k-2)$-th root of unity. For real numbers, the only solutions are $a = 1$ or $a = -1$ if $k-2$ is even.

For the second case, $(-a)^k = a^2$, we have:
\[ (-a)^k = a^2 \]
If $k$ is even, then $(-a)^k = a^k$, so we are back to the first case. If $k$ is odd, then $(-a)^k = -a^k$, so we have:
\[ -a^k = a^2 \]
\[ -a^2 (a^{k-2} - 1) = 0 \]
This gives us $a = 0$ or $a^{k-2} = -1$. For real numbers, the only solution is $a = -1$ if $k-2$ is odd.

So, for $k \geq 3$, the possible values for $a$ are $0$, $1$, and $-1$. We need to find the smallest $k$ such that there are at least 2009 distinct special $k$-tuples. The distinct $k$-tuples can be formed by choosing $a = 0$, $a = 1$, or $a = -1$ for each $x_i$. If $a = 0$, there is only one tuple, which is $(0, 0, \ldots, 0)$. If $a = 1$ or $a = -1$, we can have different combination